<a href="https://colab.research.google.com/github/APosokhova893/python_basics/blob/main/%22HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Data Analytics Course/modul 15/cookie_cats.csv')

In [4]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [5]:
import statsmodels.stats.api as sms
import math
#Початкові дані
p1 = 0.19
p2 = 0.20
#Розрахунок ефекту
effect_size = sms.proportion_effectsize(p1, p2)
#Розрахунок розміру вибірки
n = sms.NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.8,
    ratio=1 )
#Округлення до цілого числа в більшу сторону
n = math.ceil(n)
print("Кількість гравців в одній групі:", n)
print("Загальна кількість гравців:", n * 2)

Кількість гравців в одній групі: 24638
Загальна кількість гравців: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [6]:
df.groupby('version')['retention_7'].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


H₀ (нульова гіпотеза): утримання гравців через 7 днів однакове для версій gate_30 та gate_40.

H₁ (альтернативна гіпотеза): утримання гравців через 7 днів відрізняється між версіями гри.

За результатами вибірки версія gate_30 показує краще утримання (19,02%) порівняно з gate_40 (18,20%).

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [7]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
#Розділяємо гравців на дві групи
control = df[df['version'] == 'gate_30']
treatment = df[df['version'] == 'gate_40']
#Кількість гравців, які повернулися через 7 днів
count = [
    control['retention_7'].sum(),
    treatment['retention_7'].sum()
     ]
#Загальна кількість гравців у кожній групі
nobs = [
    len(control),
    len(treatment)
    ]
print("Кількість гравців, які повернулися:", count)
print("Кількість гравців у групах:", nobs)

Кількість гравців, які повернулися: [np.int64(8502), np.int64(8279)]
Кількість гравців у групах: [44700, 45489]


In [8]:
stat, pval = proportions_ztest(count, nobs)

print("Z-статистика:", stat)
print("p-value:", pval)

Z-статистика: 3.164358912748191
p-value: 0.001554249975614329


In [9]:
from statsmodels.stats.proportion import proportion_confint
ci_control = proportion_confint(count[0], nobs[0], alpha=0.05)
ci_treatment = proportion_confint(count[1], nobs[1], alpha=0.05)

print("95% довірчий інтервал control:", ci_control)
print("95% довірчий інтервал treatment:", ci_treatment)

95% довірчий інтервал control: (0.18656311652199903, 0.19383956804175934)
95% довірчий інтервал treatment: (0.17845430073314686, 0.18554578720019968)


За результатами Z-тесту отримано p-value = 0,00155, що менше за рівень значущості 0,05. Отже, нульову гіпотезу відхиляємо.

Між версіями гри є статистично значуща різниця в утриманні гравців через 7 днів. Версія gate_30 показує краще утримання (19,02%), ніж gate_40 (18,20%).

95% довірчі інтервали для двох груп не перетинаються, що узгоджується з результатом Z-тесту.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [10]:
table = pd.crosstab(df['version'], df['retention_7'])
print(table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [11]:
chi2, p_value, dof, expected = stats.chi2_contingency(table)

print("Хі-квадрат:", chi2)
print("p-value:", p_value)

Хі-квадрат: 9.959086799559167
p-value: 0.0016005742679058301


H₀: Версія гри та утримання гравців через 7 днів незалежні.

H₁: Між версією гри та утриманням гравців через 7 днів існує залежність.

Результати тесту хі-квадрат:

χ² = 9,9591

p-value = 0,00160

Рівень значущості α = 0,05.

Висновок:

Оскільки p-value (0,00160) менше за рівень значущості 0,05, нульову гіпотезу відхиляємо.

Отже, між версією гри та утриманням гравців через 7 днів існує статистично значущий зв’язок.

Версія gate_30 показала вище утримання гравців (19,02%), ніж gate_40 (18,20%).